# Discover a reaction path with AFIR + MLFF (MACE)

Find how a molecule rearranges **without knowing the product in advance**: an artificial force (AFIR) pulls two selected atoms together on top of the physical potential energy surface, while a **MACE** foundation model supplies energies and forces, so bonds break and form on the fly.

The example is the **Claisen rearrangement** of allyl vinyl ether (C5H8O) into 4-pentenal — a concerted [3,3]-sigmatropic shift in which a C–O bond breaks and a C–C bond forms in one step. Its ~30 kcal/mol barrier is far too high for a plain relaxation to cross, which is exactly the situation AFIR is built for.

<h2 style="color:green">Usage</h2>

1. Set the molecule, the reacting atom pairs and the MACE model in cells 1.2 and 1.3 (or use default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the search to finish (a few minutes on a laptop CPU).
1. Scroll down to view the discovered path, the transition state and the energy diagram.

## Summary

Load the molecule, relax it with MACE, ramp an artificial force between the two target atoms until the reaction happens, strip the bias to recover the physical energy profile, relax the discovered product, refine the highest point of the path into a true saddle point, verify it with a vibrational analysis and by relaxing along the imaginary mode in both directions, and save the reactant, transition state and product together with the energy profile and the plots.

Everything runs locally through ASE — no platform compute and no authentication.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.mlff import get_mlff_install_profiles
from mat3ra.notebooks_utils.packages import install_packages

await install_packages(get_mlff_install_profiles("mace"))

from mat3ra.notebooks_utils.pyodide.packages.patches import apply_all_patches

apply_all_patches("mace")

### 1.2. Set the reaction

The atom pairs are indices into the loaded structure. Cell 2.1 prints the element of every index so they can be checked before the search starts.

In [ ]:
FOLDER = "../../uploads"  # molecules are read from here, and the resulting materials are written back here
RESULTS_FOLDER = "results"  # energy profile, plots and the raw trajectory
MOLECULE_NAME = "allyl vinyl ether"  # looked up in the uploads folder first, then fetched from PubChem by name
PRODUCT_NAME = "4-pentenal"

# The bond that AFIR forces to form: the two terminal CH2 carbons
BOND_FORMING_PAIR = (4, 5)
# The bond expected to break in response: the ether oxygen and the allylic CH2
BOND_BREAKING_PAIR = (0, 1)
# The bond that becomes the product carbonyl
CARBONYL_PAIR = (0, 3)

### 1.3. AFIR and MACE options

AFIR adds a bias term $E_\text{bias} = \alpha \, r_{ij}$ between the target atoms, i.e. a constant attractive force $\alpha$ that pulls them together no matter how the physical surface resists. A single value of $\alpha$ either leaves the molecule stuck in a biased minimum (too weak) or drags it through a badly distorted geometry (too strong), so the force is ramped: each stage starts from the structure the previous one converged to.

In [ ]:
# Artificial force applied between the target atoms, ramped stage by stage (eV/Å)
AFIR_FORCE_RAMP = [1.0, 2.0, 3.0, 4.0]
AFIR_MAX_STEPS_PER_STAGE = 150
AFIR_MAX_DISPLACEMENT = 0.1  # per-step displacement cap, keeps the biased path smooth (Å)
AFIR_TRAJECTORY_PATH = f"{RESULTS_FOLDER}/afir_path.traj"

# Maximum force on any atom at convergence (eV/Å)
RELAXATION_FMAX = 0.03
AFIR_FMAX = 0.05
SADDLE_FMAX = 0.02

# Modes below this magnitude are the translations and rotations of a free molecule (cm⁻¹)
IMAGINARY_MODE_THRESHOLD = 50
# Displacement along the imaginary mode used to leave the saddle point (Å)
REACTION_MODE_DISPLACEMENT = 0.3
# Two relaxed structures count as different minima if a target distance differs by more than this (Å)
MINIMUM_SEPARATION = 0.5

# "mp": MACE-MP models shipped with the platform, the only option that works in JupyterLite;
# "off": MACE-OFF23 for organic molecules, downloaded on first use, so local runs only
MACE_MODEL_FAMILY = "mp"
MACE_MODEL = "medium"  # choose between "small", "medium" and "large"
MACE_DISPERSION = False  # D3 dispersion correction, not needed for this intramolecular rearrangement
MACE_DEFAULT_DTYPE = "float64"  # float64 is recommended for geometry optimization and vibrations
MACE_DEVICE = "cpu"  # hardware target: "cpu" or "cuda" (GPU)

## 2. Load the molecule
### 2.1. Read from uploads, or fetch the 3D structure from PubChem

PubChem serves an optimized 3D conformer for most small molecules, which makes any named molecule a valid starting point for the search.

In [ ]:
import io
import os
from urllib.parse import quote

from ase.io import read, write

PUBCHEM_STRUCTURE_URL = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/SDF?record_type=3d"


def fetch_pubchem_structure(name):
    url = PUBCHEM_STRUCTURE_URL.format(name=quote(name))
    try:
        from pyodide.http import open_url

        return open_url(url).read()
    except ImportError:
        from urllib.request import urlopen

        return urlopen(url).read().decode()


molecule_path = os.path.join(FOLDER, MOLECULE_NAME.replace(" ", "_") + ".xyz")

if not os.path.exists(molecule_path):
    write(molecule_path, read(io.StringIO(fetch_pubchem_structure(MOLECULE_NAME)), format="sdf"))
    print(f"Fetched {MOLECULE_NAME} from PubChem, saved to {molecule_path}")

molecule = read(molecule_path)

print(f"{MOLECULE_NAME}: {molecule.get_chemical_formula()}")
print("atoms: " + ", ".join(f"{index}:{symbol}" for index, symbol in enumerate(molecule.get_chemical_symbols())))
print(f"bond to form  {BOND_FORMING_PAIR}: {molecule.get_distance(*BOND_FORMING_PAIR):.2f} Å")
print(f"bond to break {BOND_BREAKING_PAIR}: {molecule.get_distance(*BOND_BREAKING_PAIR):.2f} Å")

### 2.2. View the molecule

In [ ]:
from mat3ra.made.tools.build_components import MaterialWithBuildMetadata
from mat3ra.made.tools.convert import from_ase
from mat3ra.notebooks_utils.ipython.entity.material.visualize import ViewersEnum, visualize_materials as visualize

VACUUM = 5.0  # padding around the molecule, in Å, so it can be handled as a material


def to_material(atoms, name):
    boxed_atoms = atoms.copy()
    boxed_atoms.center(vacuum=VACUUM)
    material = MaterialWithBuildMetadata.create(from_ase(boxed_atoms))
    material.name = name
    return material


visualize([{"material": to_material(molecule, MOLECULE_NAME), "title": MOLECULE_NAME}], viewer=ViewersEnum.wave)

## 3. Relax the reactant with MACE
### 3.1. Create the ASE calculator

With `MACE_MODEL_FAMILY = "mp"` the calculator is built from the MACE-MP models shipped with the platform (`packages/models`), so nothing is downloaded at run time and the notebook runs in JupyterLite.

Those models are trained on inorganic crystal trajectories. For this organic rearrangement they close the C–C bond but do not break the C–O bond, so the search ends at a cyclic structure rather than at 4-pentenal. `MACE_MODEL_FAMILY = "off"` selects MACE-OFF23, which is trained on organic molecules and reproduces the published mechanism — it is downloaded on first use, so it works locally but not in the browser, and its Academic Software License does not permit commercial use.

In [ ]:
from mat3ra.notebooks_utils.mlff import create_mlff_calculator

if MACE_MODEL_FAMILY == "off":
    from mace.calculators import mace_off

    calculator = mace_off(model=MACE_MODEL, default_dtype=MACE_DEFAULT_DTYPE, device=MACE_DEVICE)
else:
    calculator = create_mlff_calculator(
        "mace",
        {
            "model": MACE_MODEL,
            "dispersion": MACE_DISPERSION,
            "default_dtype": MACE_DEFAULT_DTYPE,
            "device": MACE_DEVICE,
        },
    )

### 3.2. Relax the reactant

In [ ]:
from ase.optimize import BFGS

# ASE sends optimizer logs to /dev/null when no logfile is given, and Pyodide cannot flush it; a buffer works in both
OPTIMIZER_LOG = io.StringIO()

reactant = molecule.copy()
reactant.calc = calculator

BFGS(reactant, logfile=OPTIMIZER_LOG).run(fmax=RELAXATION_FMAX)
reactant_energy = reactant.get_potential_energy()

print(f"Relaxed {MOLECULE_NAME}: {reactant_energy:.3f} eV")

## 4. Push the reaction with an artificial force

The bias is applied with ASE's `ExternalForce` constraint, which adds exactly the AFIR term: a constant force of $\alpha$ along the vector connecting the two target atoms, with an energy contribution $\alpha \, r_{ij}$. The physical forces still come from MACE, so the molecule is free to respond by breaking whatever bond is in the way.

In [ ]:
from ase.constraints import ExternalForce
from ase.io.trajectory import Trajectory

structure = reactant.copy()
structure.calc = calculator

os.makedirs(RESULTS_FOLDER, exist_ok=True)
trajectory = Trajectory(AFIR_TRAJECTORY_PATH, "w", structure)
trajectory.write()

for artificial_force in AFIR_FORCE_RAMP:
    structure.set_constraint(ExternalForce(*BOND_FORMING_PAIR, -artificial_force))
    BFGS(structure, trajectory=trajectory, maxstep=AFIR_MAX_DISPLACEMENT, logfile=OPTIMIZER_LOG).run(
        fmax=AFIR_FMAX, steps=AFIR_MAX_STEPS_PER_STAGE
    )
    print(
        f"α = {artificial_force:.1f} eV/Å  →  "
        f"d{BOND_FORMING_PAIR} = {structure.get_distance(*BOND_FORMING_PAIR):.2f} Å, "
        f"d{BOND_BREAKING_PAIR} = {structure.get_distance(*BOND_BREAKING_PAIR):.2f} Å"
    )

structure.set_constraint()

## 5. Recover the physical energy landscape
### 5.1. Strip the bias

Every structure along the biased path is re-evaluated with the bias removed, which turns the AFIR trajectory into a physical energy profile. Its maximum is the first estimate of the transition state.

In [ ]:
import numpy as np

EV_TO_KCAL_PER_MOL = 23.060548

images = read(AFIR_TRAJECTORY_PATH, index=":")
unbiased_energies = []
for image in images:
    image.set_constraint()
    image.calc = calculator
    unbiased_energies.append(image.get_potential_energy())

path_energies = (np.array(unbiased_energies) - reactant_energy) * EV_TO_KCAL_PER_MOL
transition_state_guess_index = int(np.argmax(path_energies))

print(f"AFIR path: {len(images)} structures")
print(
    f"Highest point at step {transition_state_guess_index}: "
    f"{path_energies[transition_state_guess_index]:.1f} kcal/mol above the reactant"
)

### 5.2. Plot the discovered path

In [ ]:
from matplotlib import pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

forming_distances = [image.get_distance(*BOND_FORMING_PAIR) for image in images]
breaking_distances = [image.get_distance(*BOND_BREAKING_PAIR) for image in images]

path_figure, (energy_axes, distance_axes) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

energy_axes.plot(path_energies, color="#2b5c8f", linewidth=2)
energy_axes.axvline(transition_state_guess_index, color="#c93b3b", linestyle="--", label="Transition state guess")
energy_axes.set_ylabel("Energy relative to reactant (kcal/mol)")
energy_axes.set_title(f"AFIR path: {MOLECULE_NAME} (MACE-MP, {MACE_MODEL})")
energy_axes.legend()
energy_axes.grid(True, linestyle=":", alpha=0.6)

distance_axes.plot(forming_distances, color="#2e8b57", label=f"forming {BOND_FORMING_PAIR}")
distance_axes.plot(breaking_distances, color="#c93b3b", label=f"breaking {BOND_BREAKING_PAIR}")
distance_axes.axvline(transition_state_guess_index, color="#c93b3b", linestyle="--")
distance_axes.set_xlabel("AFIR step")
distance_axes.set_ylabel("Distance (Å)")
distance_axes.legend()
distance_axes.grid(True, linestyle=":", alpha=0.6)

path_figure.tight_layout()
display_matplotlib_figure(path_figure)

## 6. Relax the discovered product

The last structure of the biased path is relaxed with the bias removed, which lets it settle into the product basin the search reached.

In [ ]:
product = images[-1].copy()
product.calc = calculator

BFGS(product, logfile=OPTIMIZER_LOG).run(fmax=RELAXATION_FMAX)
reaction_energy = (product.get_potential_energy() - reactant_energy) * EV_TO_KCAL_PER_MOL

print(f"Product energy: {reaction_energy:.1f} kcal/mol relative to the reactant\n")
print(f"{'bond':<20}{'reactant':>12}{'product':>12}")
for label, pair in (
    ("forming", BOND_FORMING_PAIR),
    ("breaking", BOND_BREAKING_PAIR),
    ("carbonyl", CARBONYL_PAIR),
):
    print(f"{label + ' ' + str(pair):<20}{reactant.get_distance(*pair):>10.2f} Å{product.get_distance(*pair):>10.2f} Å")

## 7. Refine the transition state

The maximum of the AFIR path is a point on a biased trajectory, not a stationary point of the physical surface: the forces there are still large and its energy overestimates the barrier. The dimer method follows the lowest-curvature mode uphill and all remaining modes downhill, converging on the first-order saddle point nearest that guess — that saddle is what sets the activation energy. It is started along the reaction direction: the forming pair closing, the breaking pair opening.

In [ ]:
from ase.mep import DimerControl, MinModeAtoms, MinModeTranslate

transition_state = images[transition_state_guess_index].copy()
transition_state.calc = calculator
print(f"Maximum force at the AFIR guess: {np.abs(transition_state.get_forces()).max():.2f} eV/Å")

reaction_direction = np.zeros_like(transition_state.positions)
for pair, sign in ((BOND_FORMING_PAIR, 1.0), (BOND_BREAKING_PAIR, -1.0)):
    unit_vector = transition_state.positions[pair[1]] - transition_state.positions[pair[0]]
    unit_vector /= np.linalg.norm(unit_vector)
    reaction_direction[pair[0]] += sign * unit_vector
    reaction_direction[pair[1]] -= sign * unit_vector
reaction_direction /= np.linalg.norm(reaction_direction)

dimer_control = DimerControl(
    initial_eigenmode_method="displacement",
    displacement_method="vector",
    logfile=OPTIMIZER_LOG,
    eigenmode_logfile=OPTIMIZER_LOG,
)
dimer = MinModeAtoms(transition_state, dimer_control)
dimer.displace(displacement_vector=0.05 * reaction_direction, mask=[True] * len(transition_state))

MinModeTranslate(dimer, logfile=OPTIMIZER_LOG).run(fmax=SADDLE_FMAX, steps=200)

transition_state_energy = transition_state.get_potential_energy()
activation_energy = (transition_state_energy - reactant_energy) * EV_TO_KCAL_PER_MOL
saddle_force = float(np.abs(transition_state.get_forces()).max())

print(f"Maximum force at the saddle point: {saddle_force:.3f} eV/Å")
if saddle_force > SADDLE_FMAX:
    print(f"⚠️ Not converged to {SADDLE_FMAX} eV/Å — this structure is not a transition state and the numbers below say nothing about the reaction.")
print(f"Activation energy: {activation_energy:.1f} kcal/mol")
print(
    f"d{BOND_FORMING_PAIR} = {transition_state.get_distance(*BOND_FORMING_PAIR):.2f} Å, "
    f"d{BOND_BREAKING_PAIR} = {transition_state.get_distance(*BOND_BREAKING_PAIR):.2f} Å"
)

## 8. Verify the transition state

A first-order saddle point has exactly one imaginary vibrational frequency, and its mode is the reaction coordinate. A free molecule also has six translational and rotational modes at (numerically) near-zero frequency, which appear as small imaginary values — `IMAGINARY_MODE_THRESHOLD` separates those from a genuine reaction mode.

In [ ]:
from ase.vibrations import Vibrations

vibrations = Vibrations(transition_state, name="transition_state_vibrations")
vibrations.run()
vibrations.summary()

frequencies = vibrations.get_frequencies()
imaginary_mode_indices = [
    index
    for index, frequency in enumerate(frequencies)
    if np.iscomplex(frequency) and abs(frequency.imag) > IMAGINARY_MODE_THRESHOLD
]

print(f"\nImaginary modes above {IMAGINARY_MODE_THRESHOLD} cm⁻¹: {len(imaginary_mode_indices)}")
for index in imaginary_mode_indices:
    print(f"  mode {index}: {abs(frequencies[index].imag):.0f}i cm⁻¹")

if not imaginary_mode_indices:
    print("⚠️ No imaginary mode above the threshold — this structure is not a transition state.")

reaction_mode = vibrations.get_mode(imaginary_mode_indices[0] if imaginary_mode_indices else 0)
vibrations.clean()

## 9. Confirm which minima the saddle connects

Displacing along the imaginary mode in both directions and relaxing shows what the transition state actually joins: one side must fall back to the reactant, the other into the product.

In [ ]:
connected_minima = {}

print(f"{'direction':<12}{'energy, kcal/mol':>18}{'d' + str(BOND_FORMING_PAIR):>14}{'d' + str(BOND_BREAKING_PAIR):>14}")
for sign, label in ((1.0, "forward"), (-1.0, "reverse")):
    displaced = transition_state.copy()
    displaced.positions += sign * REACTION_MODE_DISPLACEMENT * reaction_mode / np.linalg.norm(reaction_mode)
    displaced.calc = calculator
    BFGS(displaced, logfile=OPTIMIZER_LOG).run(fmax=RELAXATION_FMAX, steps=400)
    connected_minima[label] = displaced
    energy = (displaced.get_potential_energy() - reactant_energy) * EV_TO_KCAL_PER_MOL
    print(
        f"{label:<12}{energy:>18.1f}"
        f"{displaced.get_distance(*BOND_FORMING_PAIR):>12.2f} Å"
        f"{displaced.get_distance(*BOND_BREAKING_PAIR):>12.2f} Å"
    )

connects_two_minima = any(
    abs(connected_minima["forward"].get_distance(*pair) - connected_minima["reverse"].get_distance(*pair))
    > MINIMUM_SEPARATION
    for pair in (BOND_FORMING_PAIR, BOND_BREAKING_PAIR)
)
print(
    "\n✅ The imaginary mode connects two distinct minima."
    if connects_two_minima
    else "\n⚠️ Both directions relax to the same structure — the saddle does not connect a reactant and a product."
)

## 10. Results
### 10.1. Energy diagram

The experimental activation energy for this reaction is 30.6 kcal/mol in the gas phase, and the rearrangement is strongly exothermic. MACE reproduces the reaction energy well; the barrier is overestimated, as foundation models trained on near-equilibrium structures generally are in the bond-breaking region.

In [ ]:
EXPERIMENTAL_ACTIVATION_ENERGY = 30.6  # kcal/mol, gas phase

levels = [
    (MOLECULE_NAME, 0.0),
    ("transition state", activation_energy),
    (PRODUCT_NAME, reaction_energy),
]

diagram_figure, axes = plt.subplots(figsize=(7, 4.5))
axes.plot(range(len(levels)), [energy for _, energy in levels], linestyle="--", color="#999999")
for position, (label, energy) in enumerate(levels):
    axes.hlines(energy, position - 0.25, position + 0.25, color="#2b5c8f", linewidth=4)
    axes.annotate(f"{energy:.1f}", (position, energy), textcoords="offset points", xytext=(0, 10), ha="center")
axes.axhline(EXPERIMENTAL_ACTIVATION_ENERGY, color="#c93b3b", linestyle=":", label="experimental barrier")
axes.set_xticks(range(len(levels)))
axes.set_xticklabels([label for label, _ in levels])
axes.set_ylabel("Energy relative to reactant (kcal/mol)")
axes.set_title(f"{MOLECULE_NAME} → {PRODUCT_NAME} (MACE-MP, {MACE_MODEL})")
axes.legend()
axes.grid(True, axis="y", linestyle=":", alpha=0.6)
diagram_figure.tight_layout()
display_matplotlib_figure(diagram_figure)

saddle_note = "" if saddle_force <= SADDLE_FMAX else "  ⚠️ no transition state was found, see 7"
print(f"Activation energy: {activation_energy:.1f} kcal/mol (experiment: {EXPERIMENTAL_ACTIVATION_ENERGY}){saddle_note}")
print(f"Reaction energy:   {reaction_energy:.1f} kcal/mol")

### 10.2. View the reactant, transition state and product

In [ ]:
visualize(
    [
        {"material": to_material(reactant, MOLECULE_NAME), "title": MOLECULE_NAME},
        {"material": to_material(transition_state, "Transition state"), "title": "Transition state"},
        {"material": to_material(product, PRODUCT_NAME), "title": PRODUCT_NAME},
    ],
    viewer=ViewersEnum.wave,
)

## 11. Save the results
### 11.1. Hand the structures back as materials

The three structures that define the reaction are passed to the environment with `set_materials`.

In [ ]:
from mat3ra.notebooks_utils.material import set_materials

structures = (
    ("reactant", reactant, f"{MOLECULE_NAME}, reactant"),
    ("transition_state", transition_state, f"{MOLECULE_NAME} to {PRODUCT_NAME}, transition state"),
    ("product", product, f"{PRODUCT_NAME}, product"),
)

set_materials([to_material(atoms, name) for _, atoms, name in structures], FOLDER)

### 11.2. Write the energy profile and the plots

The trajectory of the search is already in `RESULTS_FOLDER`; this adds the profile behind the plots, the settings and the resulting energies, so the run can be reported without re-running it.

In [ ]:
import json

results = {
    "reaction": {"reactant": MOLECULE_NAME, "product": PRODUCT_NAME},
    "settings": {
        "calculator": f"MACE-MP {MACE_MODEL}",
        "bond_forming_pair": list(BOND_FORMING_PAIR),
        "bond_breaking_pair": list(BOND_BREAKING_PAIR),
        "artificial_force_ramp_ev_per_angstrom": AFIR_FORCE_RAMP,
    },
    "activation_energy_kcal_per_mol": round(float(activation_energy), 2),
    "reaction_energy_kcal_per_mol": round(float(reaction_energy), 2),
    "imaginary_frequency_cm": (
        round(float(abs(frequencies[imaginary_mode_indices[0]].imag)), 1) if imaginary_mode_indices else None
    ),
    "transition_state_found": bool(saddle_force <= SADDLE_FMAX and connects_two_minima),
    "distances_angstrom": {
        role: {
            "forming": round(float(atoms.get_distance(*BOND_FORMING_PAIR)), 3),
            "breaking": round(float(atoms.get_distance(*BOND_BREAKING_PAIR)), 3),
            "carbonyl": round(float(atoms.get_distance(*CARBONYL_PAIR)), 3),
        }
        for role, atoms, _ in structures
    },
    "afir_path": {
        "transition_state_guess_index": transition_state_guess_index,
        "energy_kcal_per_mol": [round(float(value), 4) for value in path_energies],
        "forming_distance_angstrom": [round(float(value), 3) for value in forming_distances],
        "breaking_distance_angstrom": [round(float(value), 3) for value in breaking_distances],
    },
}

with open(os.path.join(RESULTS_FOLDER, "reaction_path.json"), "w") as file:
    json.dump(results, file, indent=2)

path_figure.savefig(os.path.join(RESULTS_FOLDER, "afir_path.png"), dpi=140)
diagram_figure.savefig(os.path.join(RESULTS_FOLDER, "energy_diagram.png"), dpi=140)

print(f"Saved to {RESULTS_FOLDER}/: " + ", ".join(sorted(os.listdir(RESULTS_FOLDER))))

## References

[1] AFIR method: S. Maeda, K. Morokuma, "Communications: A systematic method for locating transition structures of A+B → X type reactions", J. Chem. Phys. 132, 241102 (2010). https://doi.org/10.1063/1.3457903  
[2] AFIR review: S. Maeda, K. Ohno, K. Morokuma, "Systematic exploration of the mechanism of chemical reactions: the global reaction route mapping (GRRM) strategy", Phys. Chem. Chem. Phys. 15, 3683 (2013). https://doi.org/10.1039/C3CP44063J  
[3] MACE-OFF23 organic foundation models: D. P. Kovács et al., arXiv:2312.15211. https://arxiv.org/abs/2312.15211  
[4] MACE-MP-0 materials foundation model: I. Batatia et al., arXiv:2401.00096. https://arxiv.org/abs/2401.00096  
[5] Dimer method: G. Henkelman, H. Jónsson, "A dimer method for finding saddle points on high dimensional potential surfaces using only first derivatives", J. Chem. Phys. 111, 7010 (1999). https://doi.org/10.1063/1.480097  
[6] Experimental Claisen barrier: F. W. Schuler, G. W. Murphy, "The Kinetics of the Rearrangement of Vinyl Allyl Ether", J. Am. Chem. Soc. 72, 3155 (1950). https://doi.org/10.1021/ja01163a096  
[7] ASE optimizers, constraints and vibrations: https://wiki.fysik.dtu.dk/ase/ase/optimize.html  
[8] PubChem compound "allyl vinyl ether" (CID 221523): https://pubchem.ncbi.nlm.nih.gov/compound/221523  